In [ ]:
# ============================================================
# 环境配置（Colab / 本地 Jupyter 通用）
# - Colab 用户：直接运行本单元即可
# - 本地 Jupyter 用户：同样可直接运行
# ============================================================

import sys
import subprocess


def ensure_package(package_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


for package in ["torch", "scikit-learn", "matplotlib", "numpy"]:
    ensure_package(package)

print("Environment ready.")

# AlphaCode 代码实战：从学习掌握到实习面试表达

基于论文 *Competition-Level Code Generation with AlphaCode*（Yujia Li et al., 2022），
本 Notebook 用一个 **可在 Colab 运行的教学化缩小版 AlphaCode**，帮助你完成两件事：

1. **学习路径（Learning）**：看清 AlphaCode 的模型层与系统层机制。
2. **工程路径（Engineering）**：掌握如何用更高层 API 快速搭出同类 baseline。

我们会围绕同一个 toy 任务展开：
**输入题目描述，输出简短 Python 代码；再模拟候选筛选、聚类与 n@k 评估。**

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 AlphaCode 为什么不只是“代码生成模型” | 掌握从原理到简易工程实现的迁移方式 |
| 实现方式 | 手写简化版 Encoder-Decoder + 正确性辅助信号 | `nn.Transformer` + `sklearn.cluster.KMeans` |
| 代码量 | 更多，强调 shape / mask / teacher forcing | 更少，强调 API 抽象与复用 |
| 适合场景 | 学习、复习、面试原理题 | 简单工程 demo、课程作业、baseline |

> 这不是论文级复现，而是 **面向学习与面试表达的最小可运行版本**。

In [ ]:
import math
import random
import collections
import io
import contextlib
import builtins

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 一、论文背景与任务定位

AlphaCode 面对的不是普通代码补全，而是 **编程竞赛问题求解**：

- 输入：自然语言题目描述
- 输出：候选代码程序
- 目标：不是“写得像代码”，而是“尽可能解对题”

因此，AlphaCode 的核心不是单个大模型，而是一条完整链路：

1. **编码器-解码器生成代码**
2. **利用正确性信号提升生成质量**
3. **大量候选的执行过滤**
4. **聚类去重与多样性选择**
5. **用 n@k 衡量有限提交位下的成功概率**

### 本 Notebook 覆盖什么？

- 一个最小可训练的 encoder-decoder 生成器
- 一个教学化的 correctness signal（辅助分类头）
- toy 版 public test filtering + clustering + n@k

### 本 Notebook 不覆盖什么？

- 论文中的大规模预训练
- 百万级候选采样
- 真实 sandbox 执行环境
- 完整独立 reranker 系统

> 你要把它理解成：**AlphaCode 面试级讲解版**，而不是完整工业复现。

## 二、最小必要理论

下面只保留理解代码所必需的最少公式；完整理论请回看配套 `07_AlphaCode.md`。

### 1. 序列到序列生成

给定题目描述 $x$，模型按自回归方式生成代码序列 $y$：

$$
P(y \mid x) = \prod_{t=1}^{T} P(y_t \mid y_{<t}, x)
$$

### 2. AlphaCode 训练中的两个目标

**语言建模损失（LM Loss）**：

$$
\mathcal{L}_{LM} = -\sum_t \log P_\theta(y_t \mid y_{<t}, x)
$$

**正确性辅助损失（toy Cls Loss）**：

$$
\mathcal{L}_{cls} = -\big(y \log p + (1-y) \log(1-p)\big)
$$

**总损失**：

$$
\mathcal{L}_{total} = \mathcal{L}_{LM} + \lambda \mathcal{L}_{cls}
$$

### 3. AlphaCode 的系统层思路

AlphaCode 的亮点不只是生成模型，而是：

1. 生成大量候选程序
2. 用 public tests 先过滤
3. 对剩余候选做 embedding + clustering
4. 优先从高质量簇中挑选提交

### 4. n@k 指标

如果总共有 $N$ 个候选，其中 $C$ 个正确，提交前 $k$ 个，至少有 $n$ 个正确的概率可写为：

$$
n@k = P(X \ge n), \quad X \sim \mathrm{Hypergeometric}(N, C, k)
$$

它比 `pass@k` 更强调：**在有限提交位里，正确解能否稳定出现在前列**。

## 三、组件映射表

| 论文组件 | 学习路径覆盖 | 工程路径 / API 对应 | 状态 |
|---|---|---|---|
| Encoder：读取题目描述 | 手写 `EncoderBlock` | `nn.Transformer.encoder` | Runnable |
| Decoder：自回归生成代码 | 手写 `DecoderBlock` + causal mask | `nn.Transformer.decoder` | Runnable |
| Positional Encoding | 手写正弦位置编码 | 仍需手动加入（库不自带） | Runnable |
| LM Loss | 手写 token-level cross entropy | 相同 | Runnable |
| Correctness signal | 手写辅助分类头 + 正负样本 | 只说明思想，不完整复刻 reranker | Runnable / Explain-only |
| Public test filtering | toy evaluator | 同一流程 | Illustrative |
| Clustering | trigram embedding + KMeans | `sklearn.cluster.KMeans` | Runnable |
| n@k | 手写超几何分布公式 | 相同 | Runnable |
| 百万级采样 / 真实 sandbox | 只做概念说明 | 无成熟单库可一键复刻 | Explain-only |

> 面试时可以直接用这张表说明：**哪些是我亲手实现的，哪些是工程抽象替我封装掉的，哪些只是论文思想演示。**

In [ ]:
# ── 共享超参数（两条路径共用） ──
SRC_LEN = 72
TGT_LEN = 72
MAX_GEN_LEN = 48

D_MODEL = 48
NUM_HEADS = 4
ENC_LAYERS = 1
DEC_LAYERS = 2
D_FF = 96
DROPOUT = 0.1

LR = 3e-3
NUM_EPOCHS = 120
BATCH_SIZE = 5
LAMBDA_CLS = 0.5
PRINT_EVERY = 30

# ── toy 数据：题目描述 -> 代码 ──
DATASET = [
    {"desc": "read two integers and print their sum",
     "code": "a,b=map(int,input().split())\nprint(a+b)",
     "tests": [("3 5", "8"), ("10 20", "30"), ("-1 1", "0")]},
    {"desc": "read two integers and print their product",
     "code": "a,b=map(int,input().split())\nprint(a*b)",
     "tests": [("3 5", "15"), ("2 7", "14"), ("0 9", "0")]},
    {"desc": "read an integer and print its square",
     "code": "n=int(input())\nprint(n*n)",
     "tests": [("4", "16"), ("0", "0"), ("-3", "9")]},
    {"desc": "read an integer and print its absolute value",
     "code": "n=int(input())\nprint(abs(n))",
     "tests": [("5", "5"), ("-3", "3"), ("0", "0")]},
    {"desc": "read two integers and print the larger one",
     "code": "a,b=map(int,input().split())\nprint(max(a,b))",
     "tests": [("3 5", "5"), ("10 2", "10"), ("4 4", "4")]},
    {"desc": "read two integers and print the smaller one",
     "code": "a,b=map(int,input().split())\nprint(min(a,b))",
     "tests": [("3 5", "3"), ("10 2", "2"), ("4 4", "4")]},
    {"desc": "read an integer and check if it is even",
     "code": "n=int(input())\nprint('YES' if n%2==0 else 'NO')",
     "tests": [("4", "YES"), ("3", "NO"), ("0", "YES")]},
    {"desc": "read an integer and check if it is positive",
     "code": "n=int(input())\nprint('YES' if n>0 else 'NO')",
     "tests": [("5", "YES"), ("-1", "NO"), ("0", "NO")]},
    {"desc": "read three integers and print their sum",
     "code": "a,b,c=map(int,input().split())\nprint(a+b+c)",
     "tests": [("1 2 3", "6"), ("0 0 0", "0")]},
    {"desc": "read an integer n and print n doubled",
     "code": "n=int(input())\nprint(n*2)",
     "tests": [("5", "10"), ("0", "0"), ("-3", "-6")]},
]


class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set(''.join(texts)))
        self.stoi = {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3}
        for ch in chars:
            if ch not in self.stoi:
                self.stoi[ch] = len(self.stoi)
        self.itos = {idx: token for token, idx in self.stoi.items()}
        self.vocab_size = len(self.stoi)

    def encode(self, text, add_bos=True, add_eos=True):
        ids = [self.stoi.get(ch, self.stoi['<UNK>']) for ch in text]
        if add_bos:
            ids = [self.stoi['<BOS>']] + ids
        if add_eos:
            ids = ids + [self.stoi['<EOS>']]
        return ids

    def decode(self, ids):
        special = {self.stoi['<PAD>'], self.stoi['<BOS>'], self.stoi['<EOS>'], self.stoi['<UNK>']}
        return ''.join(self.itos[idx] for idx in ids if idx not in special)


def pad_ids(ids, max_len, pad_id):
    return ids[:max_len] + [pad_id] * max(0, max_len - len(ids))


all_text = ''.join(sample['desc'] + sample['code'] for sample in DATASET)
tok = CharTokenizer([all_text])
PAD_ID = tok.stoi['<PAD>']
BOS_ID = tok.stoi['<BOS>']
EOS_ID = tok.stoi['<EOS>']
VOCAB_SIZE = tok.vocab_size

src_tensor = torch.tensor([
    pad_ids(tok.encode(sample['desc']), SRC_LEN, PAD_ID) for sample in DATASET
], dtype=torch.long)

tgt_tensor = torch.tensor([
    pad_ids(tok.encode(sample['code']), TGT_LEN, PAD_ID) for sample in DATASET
], dtype=torch.long)

train_ds = TensorDataset(src_tensor, tgt_tensor)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f"Vocab size: {VOCAB_SIZE}")
print(f"src_tensor shape: {tuple(src_tensor.shape)}")
print(f"tgt_tensor shape: {tuple(tgt_tensor.shape)}")
print("Example description:", DATASET[0]["desc"])
print("Example code:")
print(DATASET[0]["code"])

src_batch, tgt_batch = next(iter(train_loader))
print(f"Batch src shape: {tuple(src_batch.shape)}")
print(f"Batch tgt shape: {tuple(tgt_batch.shape)}")

## 四、学习路径：从零实现一个教学化 AlphaCode

这一部分是“看懂原理”的核心。

我们按数据流顺序实现：

1. **Embedding + Positional Encoding**
2. **Encoder**：读取题目描述
3. **Decoder**：基于题意自回归生成代码
4. **LM Loss**：学会“下一个 token 是谁”
5. **Correctness signal**：额外学习“这段代码是否像正确解”

### 为什么这里值得手写？

因为面试时常见问题不是“你会不会调库”，而是：

- causal mask 为什么必要？
- cross-attention 在哪里发生？
- teacher forcing 和真实推理有什么区别？
- AlphaCode 为什么不能只看生成模型？

手写完这一版，你就能把这些问题讲清楚。

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def causal_mask(size, target_device):
    # True 表示该位置被 mask，不允许看到未来 token
    return torch.triu(
        torch.ones(size, size, dtype=torch.bool, device=target_device),
        diagonal=1,
    )


def source_to_tensor(text):
    ids = pad_ids(tok.encode(text), SRC_LEN, PAD_ID)
    return torch.tensor([ids], dtype=torch.long, device=device)


def show_predictions(title, rows):
    print(f"\n{title}")
    print("-" * len(title))
    for row in rows:
        print(f"Description: {row['desc']}")
        print("Target code:")
        print(row["target"])
        print(f"{row['name']} prediction:")
        print(row["pred"])
        print("-" * 80)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)                    # (max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float() # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))          # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)  # (batch, seq, d_model)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_pad_mask):
        attn_out, _ = self.self_attn(
            x, x, x,
            key_padding_mask=src_pad_mask,
            need_weights=False,
        )  # (batch, src_len, d_model)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)  # (batch, src_len, d_model)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask, tgt_pad_mask, memory_pad_mask):
        self_attn_out, _ = self.self_attn(
            x, x, x,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_pad_mask,
            need_weights=False,
        )  # (batch, tgt_len, d_model)
        x = self.norm1(x + self.dropout(self_attn_out))

        cross_out, _ = self.cross_attn(
            x, memory, memory,
            key_padding_mask=memory_pad_mask,
            need_weights=False,
        )  # (batch, tgt_len, d_model)
        x = self.norm2(x + self.dropout(cross_out))

        ffn_out = self.ffn(x)  # (batch, tgt_len, d_model)
        x = self.norm3(x + self.dropout(ffn_out))
        return x


class LearningAlphaCodeModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, enc_layers, dec_layers, d_ff, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model

        self.embed = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(enc_layers)
        ])
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(dec_layers)
        ])

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        self.cls_head = nn.Linear(d_model, 1)

    def embed_tokens(self, token_ids):
        x = self.embed(token_ids) * math.sqrt(self.d_model)  # (batch, seq, d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        return x

    def encode(self, src_ids):
        src_pad_mask = src_ids.eq(PAD_ID)                    # (batch, src_len)
        x = self.embed_tokens(src_ids)                       # (batch, src_len, d_model)
        for block in self.encoder_blocks:
            x = block(x, src_pad_mask)
        return x, src_pad_mask

    def decode(self, tgt_ids, memory, memory_pad_mask):
        tgt_pad_mask = tgt_ids.eq(PAD_ID)                    # (batch, tgt_len)
        tgt_mask = causal_mask(tgt_ids.size(1), tgt_ids.device)  # (tgt_len, tgt_len)
        x = self.embed_tokens(tgt_ids)                       # (batch, tgt_len, d_model)
        for block in self.decoder_blocks:
            x = block(x, memory, tgt_mask, tgt_pad_mask, memory_pad_mask)
        return x

    def forward(self, src_ids, tgt_ids):
        memory, src_pad_mask = self.encode(src_ids)
        dec_out = self.decode(tgt_ids, memory, src_pad_mask)
        logits = self.lm_head(dec_out)                       # (batch, tgt_len, vocab_size)

        pooled = dec_out.mean(dim=1)                         # (batch, d_model)
        cls_logit = self.cls_head(pooled).squeeze(-1)        # (batch,)
        return logits, cls_logit


@torch.no_grad()
def greedy_generate(model, desc_text, max_len=MAX_GEN_LEN):
    model.eval()
    src_ids = source_to_tensor(desc_text)
    memory, src_pad_mask = model.encode(src_ids)

    generated = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        dec_out = model.decode(generated, memory, src_pad_mask)
        next_token = model.lm_head(dec_out[:, -1, :]).argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == EOS_ID:
            break

    return tok.decode(generated[0].tolist())

### 训练 vs 推理的区别

这是面试中最容易被追问的部分。

| 阶段 | 学习路径行为 |
|---|---|
| 训练 | 使用 **teacher forcing**：解码器看到真实代码前缀，目标是预测下一个 token |
| 推理 | 使用 **自回归生成**：解码器只能看到自己已经生成出的前缀 |
| mask | 两者都需要 causal mask，避免“偷看未来 token” |
| 正确性信号 | 训练阶段加入辅助分类头；推理阶段只保留生成分支 |

一句话理解：
**训练像老师带着写，推理像自己一步步往下写。**

In [ ]:
def train_learning_model(model, dataloader, num_epochs, lr, lambda_cls):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"lm": [], "cls": [], "total": []}

    for epoch in range(num_epochs):
        model.train()
        epoch_lm = 0.0
        epoch_cls = 0.0
        epoch_total = 0.0

        for src_ids, tgt_ids in dataloader:
            src_ids = src_ids.to(device)
            tgt_ids = tgt_ids.to(device)

            tgt_in = tgt_ids[:, :-1]
            tgt_out = tgt_ids[:, 1:]

            logits_pos, cls_pos = model(src_ids, tgt_in)

            # 构造负样本：打乱代码前缀，模拟“题意与代码不匹配”
            neg_tgt_in = torch.roll(tgt_in, shifts=1, dims=0)
            _, cls_neg = model(src_ids, neg_tgt_in)

            lm_loss = F.cross_entropy(
                logits_pos.reshape(-1, model.vocab_size),
                tgt_out.reshape(-1),
                ignore_index=PAD_ID,
            )

            cls_logits = torch.cat([cls_pos, cls_neg], dim=0)
            cls_targets = torch.cat([
                torch.ones_like(cls_pos),
                torch.zeros_like(cls_neg),
            ], dim=0)
            cls_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)

            total_loss = lm_loss + lambda_cls * cls_loss

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_lm += lm_loss.item()
            epoch_cls += cls_loss.item()
            epoch_total += total_loss.item()

        num_batches = len(dataloader)
        history["lm"].append(epoch_lm / num_batches)
        history["cls"].append(epoch_cls / num_batches)
        history["total"].append(epoch_total / num_batches)

        if (epoch + 1) % PRINT_EVERY == 0 or epoch == 0 or epoch == num_epochs - 1:
            print(
                f"Epoch {epoch+1:>3} | "
                f"LM {history['lm'][-1]:.4f} | "
                f"Cls {history['cls'][-1]:.4f} | "
                f"Total {history['total'][-1]:.4f}"
            )

    return history


learning_model = LearningAlphaCodeModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
).to(device)

print(f"Learning model parameters: {count_parameters(learning_model):,}")
learning_history = train_learning_model(
    learning_model,
    train_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    lambda_cls=LAMBDA_CLS,
)

learning_rows = []
for sample in DATASET[:3]:
    learning_rows.append({
        "desc": sample["desc"],
        "target": sample["code"],
        "name": "Learning",
        "pred": greedy_generate(learning_model, sample["desc"]),
    })

show_predictions("Learning path generation demo", learning_rows)

## 五、工程路径：用高层 API 快速搭出同类 baseline

现实里并没有一个“AlphaCode 官方工业库”可让你一键复现整条流程。

因此，这里的工程路径采用 **E2：有限工具链工程实现**：

- 用 `nn.Transformer` 压缩 encoder-decoder 主干
- 继续手动加 positional encoding（因为库本身不包含）
- 用 `sklearn.cluster.KMeans` 处理聚类

这条路径的价值在于：

- 你能快速搭一个可运行 baseline
- 你能说明哪些部分是库帮你封装的
- 你能更清楚 AlphaCode 的难点其实在系统链路，而不只在模型类本身

### 学习路径实现 vs 工程 API 对应关系

| 学习路径（手写） | 工程路径（高层 API） | 说明 |
|---|---|---|
| `EncoderBlock` | `nn.Transformer.encoder` | 注意力、残差、归一化被封装 |
| `DecoderBlock` | `nn.Transformer.decoder` | self-attn / cross-attn 被封装 |
| `causal_mask()` | `nn.Transformer.generate_square_subsequent_mask()` | 原理相同，只是调用方式不同 |
| `greedy_generate()` | 仍需手写 | `nn.Transformer` 不像 HuggingFace 那样自带 `generate()` |
| 手写 trigram embedding + KMeans | `sklearn.cluster.KMeans` | 聚类流程可直接复用成熟实现 |

> 面试时这里可以直接说：**工程路径不是“不懂原理”，而是在理解原理后选择更高抽象层。**

In [ ]:
class EngineeringAlphaCodeModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, enc_layers, dec_layers, d_ff, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model

        self.embed = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=num_heads,
            num_encoder_layers=enc_layers,
            num_decoder_layers=dec_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

    def embed_tokens(self, token_ids):
        x = self.embed(token_ids) * math.sqrt(self.d_model)  # (batch, seq, d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        return x

    def encode(self, src_ids):
        src_pad_mask = src_ids.eq(PAD_ID)                    # (batch, src_len)
        src_emb = self.embed_tokens(src_ids)                 # (batch, src_len, d_model)
        memory = self.transformer.encoder(
            src_emb,
            src_key_padding_mask=src_pad_mask,
        )
        return memory, src_pad_mask

    def decode(self, tgt_ids, memory, memory_pad_mask):
        tgt_pad_mask = tgt_ids.eq(PAD_ID)                    # (batch, tgt_len)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt_ids.size(1), device=tgt_ids.device
        )
        tgt_emb = self.embed_tokens(tgt_ids)                 # (batch, tgt_len, d_model)
        dec_out = self.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=memory_pad_mask,
        )
        return dec_out

    def forward(self, src_ids, tgt_ids):
        memory, src_pad_mask = self.encode(src_ids)
        dec_out = self.decode(tgt_ids, memory, src_pad_mask)
        logits = self.lm_head(dec_out)                       # (batch, tgt_len, vocab_size)
        return logits


def train_engineering_model(model, dataloader, num_epochs, lr):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"lm": []}

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for src_ids, tgt_ids in dataloader:
            src_ids = src_ids.to(device)
            tgt_ids = tgt_ids.to(device)

            tgt_in = tgt_ids[:, :-1]
            tgt_out = tgt_ids[:, 1:]

            logits = model(src_ids, tgt_in)
            loss = F.cross_entropy(
                logits.reshape(-1, model.vocab_size),
                tgt_out.reshape(-1),
                ignore_index=PAD_ID,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        history["lm"].append(epoch_loss / len(dataloader))

        if (epoch + 1) % PRINT_EVERY == 0 or epoch == 0 or epoch == num_epochs - 1:
            print(f"Epoch {epoch+1:>3} | LM {history['lm'][-1]:.4f}")

    return history


engineering_model = EngineeringAlphaCodeModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
).to(device)

print(f"Engineering model parameters: {count_parameters(engineering_model):,}")
engineering_history = train_engineering_model(
    engineering_model,
    train_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
)

engineering_rows = []
for sample in DATASET[:3]:
    engineering_rows.append({
        "desc": sample["desc"],
        "target": sample["code"],
        "name": "Engineering",
        "pred": greedy_generate(engineering_model, sample["desc"]),
    })

show_predictions("Engineering path generation demo", engineering_rows)

## 六、系统层：候选筛选、聚类与 n@k

这是 AlphaCode 与普通 seq2seq 教学模型最不一样的地方。

它的思路不是“生成一个答案就结束”，而是：

1. 生成很多候选
2. 用 public tests 过滤明显错误解
3. 对剩余候选做 embedding + clustering
4. 根据簇大小与多样性决定更优提交策略

> 为了保证 Colab 安全与稳定，这里只执行 **预先写死的 toy 候选代码**，用于演示筛选流程；并不执行模型自由生成出的任意代码。

In [ ]:
def run_toy_program(code, test_input):
    inputs = iter(test_input.strip().splitlines())
    buffer = io.StringIO()

    safe_builtins = {
        "print": lambda *args, **kwargs: builtins.print(*args, file=buffer, **kwargs),
        "input": lambda *args: next(inputs),
        "int": int,
        "map": map,
        "abs": abs,
        "max": max,
        "min": min,
    }

    exec(code, {"__builtins__": safe_builtins}, {})
    return buffer.getvalue().strip()


def passes_all_tests(code, tests):
    try:
        return all(run_toy_program(code, inp) == out for inp, out in tests)
    except Exception:
        return False


problem = DATASET[0]
correct_variants = [
    "a,b=map(int,input().split())\nprint(a+b)",
    "x,y=map(int,input().split())\nprint(x+y)",
    "a,b=map(int,input().split())\nprint(a + b)",
]
wrong_variants = [
    "a,b=map(int,input().split())\nprint(a-b)",
    "a,b=map(int,input().split())\nprint(a*b)",
    "a,b=map(int,input().split())\nprint(a+b+1)",
    "n=int(input())\nprint(n)",
]

candidates = []
for _ in range(40):
    if random.random() < 0.6:
        candidates.append(random.choice(correct_variants))
    else:
        candidates.append(random.choice(wrong_variants))

public_tests = problem["tests"][:1]
hidden_tests = problem["tests"][1:]
filtered = [code for code in candidates if passes_all_tests(code, public_tests)]
correct_after_hidden = [code for code in filtered if passes_all_tests(code, hidden_tests)]

print("Problem description:", problem["desc"])
print(f"Total candidates: {len(candidates)}")
print(f"Passed public tests: {len(filtered)}")
print(f"Passed hidden tests: {len(correct_after_hidden)}")


def code_to_embedding(code, dim=32):
    trigrams = [code[i:i+3] for i in range(max(0, len(code) - 2))]
    vec = np.zeros(dim, dtype=np.float32)
    for tg in trigrams:
        vec[hash(tg) % dim] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


if filtered:
    embeddings = np.array([code_to_embedding(code) for code in filtered])
    n_clusters = max(1, min(3, len(set(filtered))))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    cluster_sizes = collections.Counter(cluster_labels.tolist())
    print("Cluster sizes:", cluster_sizes)
else:
    embeddings = np.zeros((0, 32), dtype=np.float32)
    cluster_labels = np.array([])
    print("No candidates survived filtering.")

In [ ]:
def n_at_k(total_candidates, correct_count, n, k):
    from math import comb

    if total_candidates == 0 or k > total_candidates or n > k:
        return 0.0

    denom = comb(total_candidates, k)
    prob_less_than_n = 0.0
    for i in range(n):
        if correct_count >= i and (total_candidates - correct_count) >= (k - i):
            prob_less_than_n += comb(correct_count, i) * comb(total_candidates - correct_count, k - i) / denom
    return 1.0 - prob_less_than_n


print("=== n@k Demo ===")
print(f"n@1 = {n_at_k(len(filtered), len(correct_after_hidden), 1, 1):.4f}")
if len(filtered) >= 3:
    print(f"n@3 = {n_at_k(len(filtered), len(correct_after_hidden), 1, 3):.4f}")
else:
    print("Not enough filtered candidates for n@3 demo.")

if len(filtered) > 1 and len(set(cluster_labels.tolist())) > 1:
    coords = PCA(n_components=2).fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(7, 5))
    for cluster_id in sorted(set(cluster_labels.tolist())):
        mask = cluster_labels == cluster_id
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=70,
            alpha=0.8,
            label=f"Cluster {cluster_id}",
        )

    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_title("Candidate Cluster View")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough clustered candidates for PCA visualization.")

## 七、学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能看到 attention、mask、cross-attention、辅助损失的具体写法 | 能理解高层 API 如何压缩样板代码 |
| 代码量 | 更多，适合逐层拆解 | 更少，适合快速搭 baseline |
| 灵活性 | 可以自由改 block、loss、辅助头 | 受 `nn.Transformer` 封装边界约束 |
| 稳定性 | 更容易写错 shape 或 mask | API 更成熟，主干训练更稳 |
| 工业适配度 | 适合研究原型、面试复盘、机制验证 | 适合课程作业、原型开发、简单工程实验 |
| 适用场景 | 学习原理、准备面试、讲清模型机制 | 需要更快落地一个能跑的方案 |

> 一句话总结：**学习路径帮你“讲清楚为什么”，工程路径帮你“快速做出来”。**

## 八、结果、边界与面试表达

### 结果与边界

这个 Notebook 的价值不在于“把 AlphaCode 论文完整复刻出来”，而在于建立三层认知：

1. **模型层**：encoder-decoder 如何把题意映射到代码
2. **训练层**：teacher forcing、causal mask、LM loss、correctness signal 的作用
3. **系统层**：为什么 AlphaCode 真正强在“生成 + 过滤 + 聚类 + 排序”

但它也有明确边界：

- 这里只是 toy 数据，不是真实 CodeContests
- correctness signal 是教学化版本，不等价于论文完整 reranker
- 候选执行与聚类都是缩小版演示，不代表论文真实规模

### 高频面试题

**Q1：AlphaCode 的核心不是更大的模型，而是什么？**

核心是完整系统链路：
- 生成大量候选
- 用 public tests 快速过滤
- 用 clustering 保留多样性
- 在有限提交位下提升成功率

**Q2：为什么 AlphaCode 更适合用 encoder-decoder，而不是纯 decoder-only？**

- 竞赛题面通常较长，需要先完整理解题意
- encoder 负责题意建模，decoder 负责自回归生成代码
- 两部分分工更清晰，也更方便做不对称设计

**Q3：teacher forcing 的作用是什么？**

- 训练时让 decoder 看见真实前缀
- 能稳定学习下一个 token
- 避免一开始就被自己错误生成的 token 带偏

**Q4：public tests 为什么不能代表最终正确？**

- public tests 通常覆盖有限
- 错误程序也可能“卡过样例”
- 所以它更像第一轮过滤，而不是最终裁判

**Q5：为什么 clustering 有意义？**

- 如果大量候选都聚到某个簇，说明这类解法是模型的高概率区域
- 聚类可以减少重复提交，提高有限提交位的多样性
- 这就是 AlphaCode 系统层的重要启发式

**Q6：pass@k 和 n@k 的区别是什么？**

- `pass@k`：前 k 个里至少一个正确即可
- `n@k`：前 k 个里至少有 n 个正确，更强调排序质量和稳定性
- AlphaCode 更贴近竞赛提交限制，因此更强调 n@k

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | AlphaCode 的亮点是什么？ | 不是单个模型，而是生成、过滤、聚类、排序组成的系统。 |
| 2 | encoder 的作用是什么？ | 把长题意编码成可供 decoder 查询的语义表示。 |
| 3 | decoder 的作用是什么？ | 基于题意表示按 token 自回归生成代码。 |
| 4 | causal mask 为什么必要？ | 防止模型在训练时偷看未来 token。 |
| 5 | clustering 的意义？ | 去重并保留多样性，提高有限提交位的命中率。 |
| 6 | 为什么要讲 correctness signal？ | 因为 AlphaCode 关注的是“解对题”，不只是“像代码”。 |

### 项目表达模板

> 如果面试官问“你做过什么相关项目”，可以这样回答：

- **学习深度**：我从零实现了一个教学化 AlphaCode，包括 encoder-decoder、teacher forcing、causal mask 和辅助正确性信号。
- **工程能力**：我又用 `nn.Transformer` 和 `sklearn` 做了一个更高层的工程 baseline，说明我能从原理迁移到简单工程实现。
- **系统思维**：我能解释 AlphaCode 不是只靠生成模型，而是靠候选过滤、聚类和 n@k 这样的系统设计提升竞赛表现。

### 进阶探索方向

- 把 toy 数据换成更真实的小型竞赛数据
- 加入 beam search / temperature 采样，对比候选多样性
- 单独训练 reranker，而不是只用辅助分类头
- 用更强的代码 embedding 代替 trigram 哈希特征